In [ ]:
##modules
#%matplotlib widget
#%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs
from statsmodels.tsa.stattools import acf

import numpy as np

import pandas as pd


from joblib import Parallel, delayed

In [ ]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")

    

In [ ]:
subjects_BV = []

# Define modality pattern
if modality == "visual":
    pattern_modality = "_vis_"
elif modality == "auditory":
    pattern_modality = "_aud_"
else:
    pattern_modality = None

# Read subject names from .vhdr files in export_generic_data
for archivo in export_generic_data.glob("*.vhdr"):
    nombre = archivo.stem  # filename without extension

    # Keep only files matching the selected modality
    if pattern_modality is not None and pattern_modality in nombre:
        sujeto = nombre.split("_")[0].lower()  # e.g. s01b_vis_c_BV_mne -> s01b
        subjects_BV.append(sujeto)

# Remove duplicates and sort ignoring case
subjects_BV = sorted(set(subjects_BV), key=str.lower)

print("Subjects found:")
print(subjects_BV)




# --------------------------------------------------
# Channels: read them from the CSV created before
# --------------------------------------------------
channels = pd.read_csv(channels_structure_path / f"channels_{modality}.csv")

# Keep EEG channels in original order
channels_eeg = channels.loc[channels["type"] == "eeg", "channel"].tolist()

# Optional: get EOG channels too
channels_eog = channels.loc[channels["type"] == "eog", "channel"].tolist()

print("\nEEG channels:")
print(channels_eeg)

print("\nEOG channels:")
print(channels_eog)

del channels

filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")


In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.tsa.stattools import acf

def acf_epochs( subj,epochs,change_name=None ,adjusted=False, fft=True, alpha=None, 
               bartlett_confint=True, missing="none", isplot=False, picks="eeg"):
    """
    Calcula ACF y ACW (0 y 0.5) por sensor y por época, leyendo la condición
    desde epochs.event_id / epochs.events.

    Parameters
    ----------
    epochs : mne.Epochs
        Objeto de épocas que contiene múltiples condiciones.
    subj : str
        Identificador del sujeto.
    picks : str | list | None
        Canales a usar (por defecto 'eeg').

    Returns
    -------
    df : pandas.DataFrame
        Filas = épocas × sensores, con columnas: Subject, Condition, Epoch, Elect,
        acw_50_elect_all_epoch_all, acw_0_elect_all_epoch_all
    """

    # --- Selección de canales una sola vez ---
    epochs_eeg = epochs.copy().pick(picks=picks, exclude="bads")
    data_epochs = epochs_eeg.get_data()              # shape: (n_epochs, n_chans, n_times)
    channels = epochs_eeg.ch_names
    num_epochs, num_chans, _ = data_epochs.shape

    # --- Duración y lags ---
    duration = epochs_eeg.tmax - epochs_eeg.tmin    # en segundos
    sfreq = epochs_eeg.info["sfreq"]
    lags = int(np.round(duration * sfreq))

    # --- Condición por época (nombre legible) ---

    # --- Conditions per epoch from 2-digit trigger ---
    epoch_codes = epochs_eeg.events[:, 2]

    Condition_self = []
    Condition_emotion = []
    Condition_gaze = []

    for code in epoch_codes:
        code_str = str(code).zfill(2)   # ensure 2-digit format (e.g., 14, 25, 36)
        first_digit = int(code_str[0])  # self/gaze information
        second_digit = int(code_str[1]) # emotion information

        # SELF Condition
        if first_digit in [1, 2, 3]:
            self_label = "self"
        elif first_digit in [4, 5, 6]:
            self_label = "friend"
        elif first_digit in [7, 8, 9]:
            self_label = "unknown"
        else:
            self_label = np.nan

        # GAZE direction
        if first_digit in [1, 4, 7]:
            gaze_label = 1
        elif first_digit in [2, 5, 8]:
            gaze_label = 2
        elif first_digit in [3, 6, 9]:
            gaze_label = 3
        else:
            gaze_label = np.nan

        # EMOTION Condition
        if second_digit == 4:
            emotion_label = "positive"
        elif second_digit == 5:
            emotion_label = "neutral"
        elif second_digit == 6:
            emotion_label = "negative"
        else:
            emotion_label = np.nan

        Condition_self.append(self_label)
        Condition_emotion.append(emotion_label)
        Condition_gaze.append(gaze_label)


    # --- Función paralela por sensor ---
    def compute_acf_acw_sensor(data_sensor):
        acf_vals, qstat_vals, pvals = acf(
            data_sensor,
            adjusted=adjusted,
            fft=fft,
            qstat=True,
            nlags=lags,
            alpha=alpha,
            bartlett_confint=bartlett_confint,
            missing=missing
        )
        # ACW 0.5
        idx50 = np.where(acf_vals <= 0.5)[0]
        acw_50_lags = int(idx50[0]) if idx50.size else len(acf_vals) - 1
        acw_50_s = acw_50_lags / sfreq
        # ACW 0
        idx0 = np.where(acf_vals <= 0.0)[0]
        acw_0_lags = int(idx0[0]) if idx0.size else len(acf_vals) - 1
        acw_0_s = acw_0_lags / sfreq
        acf_mean = float(np.mean(acf_vals))
        return acf_vals, acf_mean, acw_50_s, acw_0_s

    # --- Bucle por época (paralelizando por sensor dentro) ---
    acw_50_all = []
    acw_0_all = []

    for j in range(num_epochs):
        results = Parallel(n_jobs=20)(
            delayed(compute_acf_acw_sensor)(data_epochs[j, i])
            for i in range(num_chans)
        )
        # Extraer métricas por sensor
        acw_50_all.append([r[2] for r in results])
        acw_0_all.append([r[3] for r in results])

    # --- Armar DataFrame ---
    shape_tabla = num_epochs * num_chans
    df = pd.DataFrame({
        "Subject": [subj] * shape_tabla,
        "event_id": np.repeat(epoch_codes, num_chans),
        "Condition_self": np.repeat(Condition_self, num_chans),
        "Condition_emotion": np.repeat(Condition_emotion, num_chans),
        "Condition_gaze": np.repeat(Condition_gaze, num_chans),
        "Epoch": np.repeat(epochs_eeg.selection, num_chans),
        "Elect": np.tile(channels, num_epochs),
        "acw_50_elect_all_epoch_all": np.array(acw_50_all).ravel(),
        "acw_0_elect_all_epoch_all": np.array(acw_0_all).ravel(),
    })

    return df


In [ ]:
##codigo para agrupar todas las tablas

all_tables = []
type_epoch="emoc"

for i in range(0,len(subjects_BV)):
# for i in range(0,1):
    try:
        subject=subjects_BV[i]
        path_epochs= epochs_clean_path / f"{subject}_epochs_{type_epoch}-epo.fif"
        epochs = mne.read_epochs(path_epochs, preload=True)
        if filtering:
            epochs.filter(l_freq=lfreq, h_freq=hfreq)
            print(f"Filter applied to {subject}: {lfreq}-{hfreq} Hz")
            filter_applied=True
        # if layer_script=="block":
        #     change_name="fix"
        # elif layer_script=="event":
        #     change_name="begin"
        table_autocorrelation= acf_epochs(subject, epochs,change_name=None,isplot=False)
        all_tables.append(table_autocorrelation)
        del epochs
    except:
        print(f"Error en {subject}")

autocorrelation_subjects_all = pd.concat(all_tables, ignore_index=True)


In [ ]:
# def validar_epocas_por_condicion(df, expected_channels):
#     print("🔎 Validando número de canales para cada combinación (Subject, Condition, Epoch):\n")

#     # Agrupar correctamente por sujeto, condición y época
#     counts = df.groupby(["Subject", "Condition", "Epoch"])["Elect"].count().reset_index()
#     counts.rename(columns={"Elect": "NumCanales"}, inplace=True)

#     # Mostrar distribución por condición
#     for cond in counts["Condition"].unique():
#         print(f"\n📌 Condición: {cond}")
#         dist = counts[counts["Condition"] == cond]["NumCanales"].value_counts()
#         print(dist)
#         # Verificar si hay valores inconsistentes
#         if len(dist) == 1 and dist.index[0] == expected_channels:
#             print(f"✅ Todas las épocas de {cond} tienen exactamente {expected_channels} canales.")
#         else:
#             print(f"⚠ Atención: {cond} tiene épocas con diferentes conteos de canales.")
#             print("🔬 Detalle por Epoch:")
#             print(counts[counts["Condition"] == cond][counts["NumCanales"] != expected_channels])

#     # Validar canales por sujeto
#     print("\n🔍 Validando que todos los sujetos tienen el mismo set de canales:")
#     channels_por_sujeto = df.groupby("Subject")["Elect"].unique()
#     base = set(channels_por_sujeto.iloc[0])
#     consistentes = True
#     for sujeto, canales in channels_por_sujeto.items():
#         if set(canales) != base:
#             consistentes = False
#             print(f"⚠ Diferencias en {sujeto}: {set(canales) ^ base}")
    
#     if consistentes:
#         print("✅ Todos los sujetos tienen los mismos canales.")
#     else:
#         print("⚠ No todos los sujetos tienen los mismos canales.")

# validar_epocas_por_condicion(autocorrelation_subjects_all, expected_channels=len(channels_mag))

# def detectar_epocas_inconsistentes(df, expected_channels):
#     print("🔍 Detectando épocas con número incorrecto de canales por sujeto y condición...\n")

#     # Agrupamos por sujeto, condición y época, y contamos canales
#     counts = df.groupby(["Subject", "Condition", "Epoch"])["Elect"].count().reset_index()
#     counts.rename(columns={"Elect": "NumCanales"}, inplace=True)

#     # Seleccionamos filas que NO tienen el número esperado de canales
#     inconsistentes = counts[counts["NumCanales"] != expected_channels]

#     if inconsistentes.empty:
#         print("✅ Todos los sujetos tienen el número esperado de canales en todas las épocas.")
#     else:
#         print("⚠ Se encontraron épocas con número de canales incorrecto:\n")
#         print(inconsistentes.to_string(index=False))

#     return inconsistentes
# inconsistentes_df = detectar_epocas_inconsistentes(autocorrelation_subjects_all, expected_channels=len(channels_mag))

In [ ]:
acw_results_subjects_all= autocorrelation_subjects_all[['Subject', "event_id",'Condition_self', "Condition_emotion","Condition_gaze", 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all','acw_0_elect_all_epoch_all']]



In [ ]:
# acw_results_subjects_all_avg_elect = (
#     acw_results_subjects_all
#     .groupby(['Subject', 'event_id', 'Condition_self', 'Condition_emotion', 'Condition_gaze', 'Epoch'], as_index=False)
#     .agg({
#         'acw_50_elect_all_epoch_all': 'mean',
#         'acw_0_elect_all_epoch_all': 'mean'
#     })
# )
# acw_results_subjects_all_avg_elect

In [ ]:
acw_results_subjects_all

In [ ]:
ACW_path

In [ ]:
acw_results_subjects_all[acw_results_subjects_all["Subject"]=="s01b"]

In [ ]:
if filtering==True and filter_applied==True:
    autocorrelation_subjects_all.to_pickle(ACW_path / f"autocorrelation_subjects_all_{filter_name}_{type_epoch}_{layer_script}.pickle")
    acw_results_subjects_all.to_pickle(ACW_path / f"acw_results_subjects_all_{filter_name}_{type_epoch}_{layer_script}.pickle")
    print(f"results saved in {ACW_path} / acw_results_subjects_all_{filter_name}_{type_epoch}_{layer_script}.pickle ")
else:
    autocorrelation_subjects_all.to_pickle(ACW_path / f"autocorrelation_subjects_all_{type_epoch}_{layer_script}.pickle")
    acw_results_subjects_all.to_pickle(ACW_path / f"acw_results_subjects_all_{type_epoch}_{layer_script}.pickle")
    print(f"results saved in {ACW_path} / acw_results_subjects_all_{type_epoch}_{layer_script}.pickle")


In [ ]:
acw_results_subjects_all